In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Concatenate, GlobalAveragePooling2D, Lambda, LSTM, Reshape
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split
import os

# Load handcrafted features
df = pd.read_csv("../Dataset/Features/signature_features.csv")
labels = df["label"].values  # 0 = Genuine, 1 = Forged

# Image paths
def get_path(filename, label):
    return f"../Dataset/Processing/AugmentedDataset/{'Genuine' if label == 0 else 'Forged'}/{filename}"
image_paths = [get_path(file, label) for file, label in zip(df["filename"], labels)]

# Handcrafted features
handcrafted_features = df.iloc[:, 2:].values  # Exclude filename & label

# Split dataset
train_img_paths, val_img_paths, train_labels, val_labels, train_features, val_features = train_test_split(
    image_paths, labels, handcrafted_features, test_size=0.2, random_state=42, stratify=labels)

# Image settings
image_size = (299, 299)
feature_dim = handcrafted_features.shape[1]

# Load InceptionV3 (CNN Feature Extractor)
base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(299, 299, 3))
cnn_output = GlobalAveragePooling2D()(base_model.output)
cnn_model = Model(inputs=base_model.input, outputs=cnn_output, name="CNN_Extractor")

# Data Generator with Hard Negative Mining
class SiameseDataGenerator(Sequence):
    def __init__(self, image_paths, labels, handcrafted_features, batch_size=32):
        self.image_paths = np.array(image_paths)
        self.labels = np.array(labels)
        self.handcrafted_features = np.array(handcrafted_features)
        self.batch_size = batch_size
        self.genuine_indices = np.where(labels == 0)[0]
        self.forged_indices = np.where(labels == 1)[0]
    
    def __len__(self):
        return int(np.floor(len(self.image_paths) / self.batch_size))
    
    def __getitem__(self, index):
        batch_anchors, batch_positives, batch_negatives, batch_labels = [], [], [], []
        for _ in range(self.batch_size):
            anchor_idx = np.random.choice(self.genuine_indices)
            positive_idx = np.random.choice(self.genuine_indices)
            negative_idx = self.hard_negative_score(anchor_idx, self.forged_indices)
            
            batch_anchors.append(self.load_image(self.image_paths[anchor_idx]))
            batch_positives.append(self.load_image(self.image_paths[positive_idx]))
            batch_negatives.append(self.load_image(self.image_paths[negative_idx]))
            batch_labels.append(1)  # 1 = Positive Pair, 0 = Negative Pair
        
        return ([np.array(batch_anchors), np.array(batch_positives), np.array(batch_negatives)], np.array(batch_labels))
    
    def hard_negative_score(self, anchor_idx, forged_indices):
        anchor_feat = self.handcrafted_features[anchor_idx]
        distances = [np.linalg.norm(anchor_feat - self.handcrafted_features[idx]) for idx in forged_indices]
        return forged_indices[np.argmin(distances)]
    
    @staticmethod
    def load_image(image_path):
        img = tf.keras.preprocessing.image.load_img(image_path, target_size=image_size)
        img = tf.keras.preprocessing.image.img_to_array(img)
        return tf.keras.applications.inception_v3.preprocess_input(img)

# Create data generators
train_generator = SiameseDataGenerator(train_img_paths, train_labels, train_features)
val_generator = SiameseDataGenerator(val_img_paths, val_labels, val_features)

# Siamese Network Model (Hybrid Approach)
def build_siamese_model():
    input_img = Input(shape=(299, 299, 3))
    input_feat = Input(shape=(feature_dim,))
    
    cnn_emb = cnn_model(input_img)
    
    lstm_input = Reshape((1, cnn_emb.shape[-1]))(cnn_emb)
    lstm_out = LSTM(64, return_sequences=False)(lstm_input)
    
    feat_dense = Dense(64, activation="relu")(input_feat)
    merged = Concatenate()([lstm_out, feat_dense])
    
    dense_out = Dense(128, activation="relu")(merged)
    dense_out = Dropout(0.3)(dense_out)
    return Model([input_img, input_feat], dense_out, name="Siamese_Branch")

siamese_branch = build_siamese_model()

# Define inputs for triplet loss training
anchor_input = Input(shape=(299, 299, 3))
pos_input = Input(shape=(299, 299, 3))
neg_input = Input(shape=(299, 299, 3))

anchor_feat = siamese_branch([anchor_input, Input(shape=(feature_dim,))])
pos_feat = siamese_branch([pos_input, Input(shape=(feature_dim,))])
neg_feat = siamese_branch([neg_input, Input(shape=(feature_dim,))])

def triplet_loss(_, embeddings):
    anchor, positive, negative = embeddings[0], embeddings[1], embeddings[2]
    pos_dist = tf.reduce_sum(tf.square(anchor - positive), axis=-1)
    neg_dist = tf.reduce_sum(tf.square(anchor - negative), axis=-1)
    loss = tf.maximum(pos_dist - neg_dist + 0.2, 0.0)  # Margin = 0.2
    return tf.reduce_mean(loss)

output = Lambda(lambda x: [x[0], x[1], x[2]], name="triplet_output")([anchor_feat, pos_feat, neg_feat])
siamese_model = Model([anchor_input, pos_input, neg_input], output)

siamese_model.compile(optimizer=Adam(learning_rate=0.0001), loss=triplet_loss)

# Train the Siamese Model
siamese_model.fit(train_generator, validation_data=val_generator, epochs=25)
siamese_model.save("../Model/siamese_signature_model.h5")

print("Siamese Network training completed!")


PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from PIL import Image
import os

# Load handcrafted features
df = pd.read_csv("../Dataset/Features/signature_features.csv")
labels = df["label"].values  # 0 = Genuine, 1 = Forged

def get_path(filename, label):
    return f"../Dataset/Processing/AugmentedDataset/{'Genuine' if label == 0 else 'Forged'}/{filename}"

image_paths = [get_path(file, label) for file, label in zip(df["filename"], labels)]
handcrafted_features = df.iloc[:, 2:].values  # Exclude filename & label

# Train-validation split
from sklearn.model_selection import train_test_split
train_img_paths, val_img_paths, train_labels, val_labels, train_features, val_features = train_test_split(
    image_paths, labels, handcrafted_features, test_size=0.2, random_state=42, stratify=labels)

# Image Transformations
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# PyTorch Dataset
class SiameseDataset(Dataset):
    def __init__(self, image_paths, labels, handcrafted_features, transform=None):
        self.image_paths = np.array(image_paths)
        self.labels = np.array(labels)
        self.handcrafted_features = torch.tensor(handcrafted_features, dtype=torch.float32)
        self.transform = transform
        self.genuine_indices = np.where(labels == 0)[0]
        self.forged_indices = np.where(labels == 1)[0]
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, index):
        anchor_idx = np.random.choice(self.genuine_indices)
        positive_idx = np.random.choice(self.genuine_indices)
        negative_idx = self.hard_negative_score(anchor_idx, self.forged_indices)
        
        anchor_img = self.load_image(self.image_paths[anchor_idx])
        positive_img = self.load_image(self.image_paths[positive_idx])
        negative_img = self.load_image(self.image_paths[negative_idx])
        
        anchor_feat = self.handcrafted_features[anchor_idx]
        positive_feat = self.handcrafted_features[positive_idx]
        negative_feat = self.handcrafted_features[negative_idx]
        
        return (anchor_img, anchor_feat), (positive_img, positive_feat), (negative_img, negative_feat)
    
    def hard_negative_score(self, anchor_idx, forged_indices):
        anchor_feat = self.handcrafted_features[anchor_idx]
        distances = [torch.norm(anchor_feat - self.handcrafted_features[idx]) for idx in forged_indices]
        return forged_indices[np.argmin(distances)]
    
    def load_image(self, image_path):
        img = Image.open(image_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

# Create DataLoaders
train_dataset = SiameseDataset(train_img_paths, train_labels, train_features, transform)
val_dataset = SiameseDataset(val_img_paths, val_labels, val_features, transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Define Siamese Network
class SiameseNetwork(nn.Module):
    def __init__(self, feature_dim):
        super(SiameseNetwork, self).__init__()
        self.cnn = models.inception_v3(pretrained=True, aux_logits=True)  # Keep aux_logits=True
        self.cnn.AuxLogits = None  # Manually disable auxiliary logits
        self.cnn.fc = nn.Identity()
        
        self.lstm = nn.LSTM(2048, 64, batch_first=True)
        self.fc1 = nn.Linear(feature_dim, 64)
        self.fc2 = nn.Linear(128, 128)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, img, handcrafted):
        cnn_out = self.cnn(img)
        if isinstance(cnn_out, tuple):  # Handle InceptionV3 output
            cnn_out = cnn_out[0]  # Extract only the final logits output
        
        lstm_input = cnn_out.unsqueeze(1)  # Reshape for LSTM
        lstm_out, _ = self.lstm(lstm_input)
        lstm_out = lstm_out[:, -1, :]

        handcrafted_out = F.relu(self.fc1(handcrafted))
        merged = torch.cat((lstm_out, handcrafted_out), dim=1)
        return self.dropout(F.relu(self.fc2(merged)))



# Define Triplet Loss
def triplet_loss(anchor, positive, negative, margin=0.2):
    pos_dist = F.pairwise_distance(anchor, positive)
    neg_dist = F.pairwise_distance(anchor, negative)
    loss = torch.clamp(pos_dist - neg_dist + margin, min=0.0)
    return loss.mean()

# Initialize Model
feature_dim = handcrafted_features.shape[1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SiameseNetwork(feature_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Training Loop
num_epochs = 2
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for (anchor, pos, neg) in progress_bar:
        anchor_img, anchor_feat = anchor
        pos_img, pos_feat = pos
        neg_img, neg_feat = neg
        
        anchor_img, pos_img, neg_img = anchor_img.to(device), pos_img.to(device), neg_img.to(device)
        anchor_feat, pos_feat, neg_feat = anchor_feat.to(device), pos_feat.to(device), neg_feat.to(device)
        
        optimizer.zero_grad()
        anchor_out = model(anchor_img, anchor_feat)
        pos_out = model(pos_img, pos_feat)
        neg_out = model(neg_img, neg_feat)
        
        loss = triplet_loss(anchor_out, pos_out, neg_out)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # Update progress bar
        progress_bar.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1}/{num_epochs}, Avg Loss: {total_loss/len(train_loader):.4f}")


# Save Model
torch.save(model.state_dict(), "../Model/siamese_signature_model.pth")
print("Siamese Network training completed!")


/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1/2:   0%|          | 0/898 [00:00<?, ?it/s]